# GRU + ResNet — Fake News Detection

1D residual conv blocks feed into a GRU layer.

In [1]:
import os
import re
import json
import pickle

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import callbacks

from tensorflow.keras.layers import Conv1D, BatchNormalization, Activation, Add


In [2]:
try:
    STOPWORDS = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    STOPWORDS = set(stopwords.words("english"))


## Hyperparameters
Edit these and re-run the notebook to try different settings.

In [3]:
PARAMS = dict(
    vocab_size=10000,
    max_len=40,
    embedding_dim=100,
    filters=64,
    kernel_size=3,
    num_blocks=2,
    gru_units=96,
    dropout=0.3,
    learning_rate=1e-3,
    batch_size=64,
    epochs=12,
)


In [4]:
DATA_DIR = os.path.join(os.getcwd(), "data")
RESULTS_DIR = os.path.join(os.getcwd(), "results")
RANDOM_STATE = 42
os.makedirs(RESULTS_DIR, exist_ok=True)


## Load & preprocess data
Place `Fake.csv`/`True.csv` in `experiments/data/`, or this will auto-download via `kagglehub` (needs a Kaggle API token).

In [5]:
def get_dataset_paths():
    fake_path = os.path.join(DATA_DIR, "Fake.csv")
    true_path = os.path.join(DATA_DIR, "True.csv")
    if os.path.exists(fake_path) and os.path.exists(true_path):
        return fake_path, true_path
    # Falls back to auto-download from Kaggle if you have not placed the CSVs
    # in experiments/data/ yourself. Needs a Kaggle API token at
    # C:\Users\<you>\.kaggle\kaggle.json (or KAGGLE_USERNAME/KAGGLE_KEY env vars).
    import kagglehub
    kaggle_path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")
    return os.path.join(kaggle_path, "Fake.csv"), os.path.join(kaggle_path, "True.csv")


def clean_text(text):
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", str(text).lower()).strip()
    tokens = [t for t in text.split() if t not in STOPWORDS]
    return " ".join(tokens)


fake_path, true_path = get_dataset_paths()
df_fake = pd.read_csv(fake_path)
df_true = pd.read_csv(true_path)
df_fake["label"] = 1  # fake
df_true["label"] = 0  # real
df = pd.concat([df_fake, df_true], ignore_index=True)
df = df.dropna(subset=["title"]).reset_index(drop=True)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
df["clean"] = df["title"].apply(clean_text)
df.head()


,title,text,subject,date,label,clean
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",1,ben stein calls 9th circuit court committed co...
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",0,trump drops steve bannon national security cou...
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",0,puerto rico expects u lift jones act shipping ...
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",1,oops trump accidentally confirmed leaked israe...
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",0,donald trump heads scotland reopen golf resort


In [6]:
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    df["clean"], df["label"], test_size=0.3,
    stratify=df["label"], random_state=RANDOM_STATE,
)
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text, y_temp, test_size=0.5,
    stratify=y_temp, random_state=RANDOM_STATE,
)

tokenizer = Tokenizer(num_words=PARAMS["vocab_size"], oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)


def to_padded(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=PARAMS["max_len"], padding="post", truncating="post")


X_train, y_train = to_padded(X_train_text), y_train.values
X_val, y_val = to_padded(X_val_text), y_val.values
X_test, y_test = to_padded(X_test_text), y_test.values

print("train:", X_train.shape, " val:", X_val.shape, " test:", X_test.shape)


train: (31428, 40)  val: (6735, 40)  test: (6735, 40)


## Build model

In [7]:
def residual_block(x, filters, kernel_size):
    shortcut = x
    y = Conv1D(filters, kernel_size, padding="same")(x)
    y = BatchNormalization()(y)
    y = Activation("relu")(y)
    y = Conv1D(filters, kernel_size, padding="same")(y)
    y = BatchNormalization()(y)
    if shortcut.shape[-1] != filters:
        shortcut = Conv1D(filters, 1, padding="same")(shortcut)
    out = Add()([shortcut, y])
    out = Activation("relu")(out)
    return out


from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam

inputs = Input(shape=(PARAMS["max_len"],))
x = Embedding(PARAMS["vocab_size"], PARAMS["embedding_dim"], input_length=PARAMS["max_len"])(inputs)
for _ in range(PARAMS["num_blocks"]):
    x = residual_block(x, PARAMS["filters"], PARAMS["kernel_size"])
x = GRU(PARAMS["gru_units"], dropout=PARAMS["dropout"])(x)
x = Dropout(PARAMS["dropout"])(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=Adam(learning_rate=PARAMS["learning_rate"]),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()


Model: "model"


__________________________________________________________________________________________________


 Layer (type)                Output Shape                 Param #   Connected to                  


 input_1 (InputLayer)        [(None, 40)]                 0         []                            


 embedding (Embedding)       (None, 40, 100)              1000000   ['input_1[0][0]']             


...

 dense (Dense)               (None, 1)                    97        ['dropout[0][0]']             


Total params: 1110561 (4.24 MB)


Trainable params: 1110049 (4.23 MB)


Non-trainable params: 512 (2.00 KB)


__________________________________________________________________________________________________


## Train

In [8]:
model_callbacks = [
    callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    callbacks.ModelCheckpoint(
        os.path.join(RESULTS_DIR, "gru_resnet_best.h5"), monitor="val_loss", save_best_only=True,
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=PARAMS["epochs"],
    batch_size=PARAMS["batch_size"],
    callbacks=model_callbacks,
    verbose=2,
)


Epoch 1/12


492/492 - 11s - loss: 0.6952 - accuracy: 0.5144 - val_loss: 0.6923 - val_accuracy: 0.5229 - 11s/epoch - 22ms/step


Epoch 2/12


C:\Users\ARKNOX\Documents\m.tech\.venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


492/492 - 9s - loss: 0.6926 - accuracy: 0.5217 - val_loss: 0.6923 - val_accuracy: 0.5229 - 9s/epoch - 19ms/step


...

492/492 - 9s - loss: 0.0477 - accuracy: 0.9852 - val_loss: 0.1274 - val_accuracy: 0.9562 - 9s/epoch - 18ms/step


Epoch 8/12


492/492 - 9s - loss: 0.0219 - accuracy: 0.9940 - val_loss: 0.4592 - val_accuracy: 0.7526 - 9s/epoch - 18ms/step


Epoch 9/12


492/492 - 9s - loss: 0.0168 - accuracy: 0.9949 - val_loss: 0.2225 - val_accuracy: 0.9501 - 9s/epoch - 18ms/step


## Evaluate & save results

In [9]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

metrics = {
    "loss": float(loss),
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1_score": float(f1_score(y_test, y_pred, zero_division=0)),
}
print("gru_resnet metrics:", metrics)

payload = {
    "model": "gru_resnet",
    "params": PARAMS,
    "history": {k: [float(v) for v in vals] for k, vals in history.history.items()},
    "metrics": metrics,
}
with open(os.path.join(RESULTS_DIR, "gru_resnet.json"), "w") as f:
    json.dump(payload, f, indent=2)

with open(os.path.join(RESULTS_DIR, "gru_resnet_tokenizer.pkl"), "wb") as f:
    pickle.dump(tokenizer, f)

print("Saved gru_resnet.json, gru_resnet_best.h5, gru_resnet_tokenizer.pkl in", RESULTS_DIR)


gru_resnet metrics: {'loss': 0.11786048859357834, 'accuracy': 0.9512991833704528, 'precision': 0.9672418835916935, 'recall': 0.9386886176554073, 'f1_score': 0.9527513684817056}
Saved gru_resnet.json, gru_resnet_best.h5, gru_resnet_tokenizer.pkl in C:\Users\ARKNOX\Documents\m.tech\experiments\results
